# IEEE 1584-2018 Incident Energy Debugging

**Bug:** Incident energy equation produces ~10²⁸ J/cm² instead of ~10-50 cal/cm²

**Location:** `arc_flash_studio/calculations/ieee_1584_2018/incident_energy.py`

**Reference:** IEEE 1584-2018, Section 4.6, Equations (3)-(6)

In [ ]:
import math
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Auto-reload when you edit source files
%load_ext autoreload
%autoreload 2

print(f"Project root: {project_root}")

Project root: /home/sramharack/arc_flash_studio


In [ ]:
# Import from your ieee_1584_2018 module
from arc_flash_studio.calculations.ieee_1584_2018 import (
    ElectrodeConfig,
    calculate_arc_flash,
    calculate_arcing_current,
    get_energy_coefficients,
    get_arcing_current_coefficients,
    equations_16_17_18_mv_interpolation,
    equation_1_intermediate_arcing_current,
    equation_13_equivalent_enclosure_size
)


print("Imports successful!")

Imports successful!


## 1. Test Case: IEEE 1584-2018 Annex D.1 (4160V MV Example)

Fill in expected values from your copy of the standard.

In [ ]:
# INPUTS from Annex D.1
Ibf = 15.0       # kA
Voc = 4.16       # kV
G = 104.0        # mm
D = 914.4        # mm (36")
T = 197.0        # ms
config = ElectrodeConfig.VCB

enc_width = 762 # mm
enc_height = 1143 # mm

# EXPECTED VALUES - Taken from Annex D.1
expected = {
    "Iarc_600": 11.117,     # kA (Taken from Eq. D.9)
    "Iarc_2700": 12.816,    # kA (Taken from Eq. D.11)  
    "Iarc_14300": 14.116,   # kA (Taken from Eq. D.13)  
    "Iarc_final": 12.979,   # kA (Taken from Eq. D.17)
    "E_600": 8.652,         # J/cm² (Taken from Eq. D.24)
    "E_2700": 11.977,       # J/cm² (Taken from Eq. D.26)
    "E_14300": 13.367,      # J/cm² (Taken from Eq. D.28)
    "E_final": 12.152,      # J/cm² (Taken from Eq. D.32)
}

print("Expected values from IEEE 1584-2018 Annex D.1",expected)

Expected values from IEEE 1584-2018 Annex D.1 {'Iarc_600': 11.117, 'Iarc_2700': 12.816, 'Iarc_14300': 14.116, 'Iarc_final': 12.979, 'E_600': 8.652, 'E_2700': 11.977, 'E_14300': 13.367, 'E_final': 12.152}


## 2. Validate the constants used in Eq. 1 from Secion 4.4

The equation to calculate the arcing current at 600V uses constants from Table 1 - Coefficients for Equation (1).

In [ ]:
k_600 = get_arcing_current_coefficients(config, 600)

print("TABLE 1 COEFFICIENTS (600V, VCB)")
print("=" * 40)
for name, value in k_600.items():
    print(f"{name}: {value}")

TABLE 1 COEFFICIENTS (600V, VCB)
k1: -0.04287
k2: 1.035
k3: -0.083
k4: 0
k5: 0
k6: -4.783e-09
k7: 1.962e-06
k8: -0.000229
k9: 0.003141
k10: 1.092


In [ ]:
k_2700 = get_arcing_current_coefficients(config, 2700)

print("TABLE 1 COEFFICIENTS (2700V, VCB)")
print("=" * 40)
for name, value in k_2700.items():
    print(f"{name}: {value}")

TABLE 1 COEFFICIENTS (2700V, VCB)
k1: 0.0065
k2: 1.001
k3: -0.024
k4: -1.557e-12
k5: 4.556e-10
k6: -4.186e-08
k7: 8.346e-07
k8: 5.482e-05
k9: -0.003191
k10: 0.9729


In [ ]:
k_14300 = get_arcing_current_coefficients(config, 14300)

print("TABLE 1 COEFFICIENTS (14,300V, VCB)")
print("=" * 40)
for name, value in k_14300.items():
    print(f"{name}: {value}")

TABLE 1 COEFFICIENTS (14,300V, VCB)
k1: 0.005795
k2: 1.015
k3: -0.011
k4: -1.557e-12
k5: 4.556e-10
k6: -4.186e-08
k7: 8.346e-07
k8: 5.482e-05
k9: -0.003191
k10: 0.9729


## 3. Step Through the Exponent

This is Equation 1 form Section 4.4

$$
I_{\mathrm{arc\_Voc}} =
10^{\left(k_1 + k_2 \lg I_{\mathrm{bf}} + k_3 \lg G\right)}
\left(
k_4 I_{\mathrm{bf}}^{6}
+ k_5 I_{\mathrm{bf}}^{5}
+ k_6 I_{\mathrm{bf}}^{4}
+ k_7 I_{\mathrm{bf}}^{3}
+ k_8 I_{\mathrm{bf}}^{2}
+ k_9 I_{\mathrm{bf}}
+ k_{10}
\right)
$$


<!-- The current implementation has:
```python
f_ibf = k4*Ibf^7 + k5*Ibf^6 + k6*Ibf^5 + k7*Ibf^4 + k8*Ibf^3 + k9*Ibf^2 + k10*Ibf
```

With k10 ≈ 0.97 and Ibf = 15, this gives k10*Ibf ≈ 14.6 added to the exponent! -->

## 3.1 Intermediate Arcing Current at 600V

In [ ]:
# Implementation by hand for Arcing Current at 600V

print("IEEE IMPLEMENTATION FOR ARCING CURRENT AT 600V - Exponent Breakdown")
print("=" * 70)

#Coefficients from Table 1 to be used for calculating immediate arcing current using Eq. 1 in Sec 4.4
im_600_arc_coeff_k1= -0.04287
im_600_arc_coeff_k2= 1.035
im_600_arc_coeff_k3= -0.083
im_600_arc_coeff_k4= 0
im_600_arc_coeff_k5= 0
im_600_arc_coeff_k6= -4.783e-09
im_600_arc_coeff_k7= 1.962e-06
im_600_arc_coeff_k8= -0.000229
im_600_arc_coeff_k9= 0.003141
im_600_arc_coeff_k10= 1.092 

# Iarc_ref = Ibf
Ibf = 15

exp_600 =  im_600_arc_coeff_k1 
print(f"k1                         = {exp_600:.6f}")

exp_600 += im_600_arc_coeff_k2 * math.log10(Ibf)  
print(f"+ k2*lg(Ibf)                 = {exp_600:.6f}")

exp_600 += im_600_arc_coeff_k3 * math.log10(G)
print(f"+ k3*lg(Ibf)          = {exp_600:.6f}")

print(f"10^exponent = {10**exp_600:.6f}")



# THE POLYNOMIAL - Current (buggy?) implementation
print("\n--- POLYNOMIAL (Hand implementation) ---")
f_ibf_600 = (
    im_600_arc_coeff_k4 * Ibf**6 +
    im_600_arc_coeff_k5 * Ibf**5 +
    im_600_arc_coeff_k6 * Ibf**4 +
    im_600_arc_coeff_k7 * Ibf**3 +
    im_600_arc_coeff_k8 * Ibf**2 +
    im_600_arc_coeff_k9 * Ibf +
    im_600_arc_coeff_k10 
)
print(f"k4*Ibf^6 = {im_600_arc_coeff_k4 * Ibf**6:.6f}")
print(f"k5*Ibf^5 = {im_600_arc_coeff_k5 * Ibf**5:.6f}")
print(f"k6*Ibf^4 = {im_600_arc_coeff_k6 * Ibf**4:.6f}")
print(f"k7*Ibf^3 = {im_600_arc_coeff_k7 * Ibf**3:.6f}")
print(f"k8*Ibf^2 = {im_600_arc_coeff_k8 * Ibf**2:.6f}")
print(f"k9*Ibf = {im_600_arc_coeff_k9 * Ibf:.6f}")
print(f"k10  = {im_600_arc_coeff_k10}")
print(f"\nTotal polynomial f(Ibf) = {f_ibf_600:.6f}")



print("\n" + "=" * 70)
print(f"result = (10^exp)*polynomial = {(10**exp_600)*f_ibf_600}")
print("\n" + "=" * 70)
im_600_result = (10**(exp_600)) * f_ibf_600

# print(f"Intermeddiate Arcing Current at 600V = {im_result:2.3f} kA")
print(f"Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {im_600_result:2.3f} kA (expected: {expected['Iarc_600']} kA)")
lib_calc_im_600_result = equation_1_intermediate_arcing_current(Ibf,G,ElectrodeConfig.VCB,600)
print(f"Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {lib_calc_im_600_result:2.3f} kA (expected: {expected['Iarc_600']} kA)")

IEEE IMPLEMENTATION FOR ARCING CURRENT AT 600V - Exponent Breakdown
k1                         = -0.042870
+ k2*lg(Ibf)                 = 1.174384
+ k3*lg(Ibf)          = 1.006971
10^exponent = 10.161801

--- POLYNOMIAL (Hand implementation) ---
k4*Ibf^6 = 0.000000
k5*Ibf^5 = 0.000000
k6*Ibf^4 = -0.000242
k7*Ibf^3 = 0.006622
k8*Ibf^2 = -0.051525
k9*Ibf = 0.047115
k10  = 1.092

Total polynomial f(Ibf) = 1.093970

result = (10^exp)*polynomial = 11.116701487164086

Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 11.117 kA (expected: 11.117 kA)
Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 11.117 kA (expected: 11.117 kA)


In [ ]:
# Energy Calculation
# exp += f_ibf
# print(f"\n+ f(Ibf)                   = {exp:.6f}")

# exp += k['k11'] * math.log10(Ibf)
# print(f"+ k11*lg(Ibf)              = {exp:.6f}")

# exp += k['k12'] * math.log10(D)
# print(f"+ k12*lg(D)                = {exp:.6f}")

# exp += k['k13'] * math.log10(Iarc_final)
# print(f"+ k13*lg(Iarc_final)       = {exp:.6f}")

# exp += math.log10(1/CF)
# print(f"+ lg(1/CF)                 = {exp:.6f}")

# E = (12.552/50) * T * (10**exp)
# print(f"\nE = {E:.4e} J/cm²")
# print(f"E = {E/4.184:.4e} cal/cm²")

## 3.2 Intermediate Arcing Current at 2700V

In [ ]:
# Implementation by hand for Arcing Current at 600V

print("IEEE IMPLEMENTATION FOR ARCING CURRENT AT 14300V - Exponent Breakdown")
print("=" * 70)

#Coefficients from Table 1 to be used for calculating immediate arcing current using Eq. 1 in Sec 4.4 @ 14300V
im_14300_arc_coeff_k1= 0.005795
im_14300_arc_coeff_k2= 1.015
im_14300_arc_coeff_k3= -0.011
im_14300_arc_coeff_k4= -1.557e-12
im_14300_arc_coeff_k5= 4.556e-10
im_14300_arc_coeff_k6= -4.186e-8
im_14300_arc_coeff_k7= 8.346e-7
im_14300_arc_coeff_k8= 5.482e-5
im_14300_arc_coeff_k9= -0.003191
im_14300_arc_coeff_k10= 0.9729


exp_14300 =  im_14300_arc_coeff_k1 
print(f"k1                         = {exp_14300:.6f}")

exp_14300 += im_14300_arc_coeff_k2 * math.log10(Ibf)  
print(f"+ k2*lg(Ibf)                 = {exp_14300:.6f}")

exp_14300 += im_14300_arc_coeff_k3 * math.log10(G)
print(f"+ k3*lg(Ibf)          = {exp_14300:.6f}")

print(f"10^exponent = {10**exp_14300:.6f}")



# THE POLYNOMIAL - Current (buggy?) implementation
print("\n--- POLYNOMIAL (Hand implementation) ---")
f_ibf_14300 = (
    im_14300_arc_coeff_k4 * Ibf**6 +
    im_14300_arc_coeff_k5 * Ibf**5 +
    im_14300_arc_coeff_k6 * Ibf**4 +
    im_14300_arc_coeff_k7 * Ibf**3 +
    im_14300_arc_coeff_k8 * Ibf**2 +
    im_14300_arc_coeff_k9 * Ibf +
    im_14300_arc_coeff_k10 
)
print(f"k4*Ibf^6 = {im_14300_arc_coeff_k4 * Ibf**6:.6f}")
print(f"k5*Ibf^5 = {im_14300_arc_coeff_k5 * Ibf**5:.6f}")
print(f"k6*Ibf^4 = {im_14300_arc_coeff_k6 * Ibf**4:.6f}")
print(f"k7*Ibf^3 = {im_14300_arc_coeff_k7 * Ibf**3:.6f}")
print(f"k8*Ibf^2 = {im_14300_arc_coeff_k8 * Ibf**2:.6f}")
print(f"k9*Ibf = {im_14300_arc_coeff_k9 * Ibf:.6f}")
print(f"k10  = {im_14300_arc_coeff_k10}")
print(f"\nTotal polynomial f(Ibf) = {f_ibf_14300:.6f}")



print("\n" + "=" * 70)
print(f"result = (10^exp)*polynomial =,{(10**exp_14300)*f_ibf_14300}")
print("\n" + "=" * 70)
im_14300_result = (10**(exp_14300)) * f_ibf_14300
print(f"Intermeddiate Arcing Current at 600V = {im_14300_result:2.3f} kA")



# print(f"Intermeddiate Arcing Current at 600V = {im_result:2.3f} kA")
print(f"Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {im_14300_result:2.3f} kA (expected: {expected['Iarc_14300']} kA)")
lib_calc_im_14300_result = equation_1_intermediate_arcing_current(Ibf,G,ElectrodeConfig.VCB,14300)
print(f"Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {lib_calc_im_14300_result:2.3f} kA (expected: {expected['Iarc_14300']} kA)")

IEEE IMPLEMENTATION FOR ARCING CURRENT AT 14300V - Exponent Breakdown
k1                         = 0.005795
+ k2*lg(Ibf)                 = 1.199528
+ k3*lg(Ibf)          = 1.177340
10^exponent = 15.043201

--- POLYNOMIAL (Hand implementation) ---
k4*Ibf^6 = -0.000018
k5*Ibf^5 = 0.000346
k6*Ibf^4 = -0.002119
k7*Ibf^3 = 0.002817
k8*Ibf^2 = 0.012334
k9*Ibf = -0.047865
k10  = 0.9729

Total polynomial f(Ibf) = 0.938395

result = (10^exp)*polynomial =,14.116469937683767

Intermeddiate Arcing Current at 600V = 14.116 kA
Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 14.116 kA (expected: 14.116 kA)
Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 14.116 kA (expected: 14.116 kA)


## 3.3 Intermeddiate Arcing Current at 14300V

In [ ]:
# Implementation by hand for Arcing Current at 600V

print("IEEE IMPLEMENTATION FOR ARCING CURRENT AT 2700V - Exponent Breakdown")
print("=" * 70)

#Coefficients from Table 1 to be used for calculating immediate arcing current using Eq. 1 in Sec 4.4 @ 2700V
im_2700_arc_coeff_k1= 0.0065
im_2700_arc_coeff_k2= 1.001
im_2700_arc_coeff_k3= -0.024
im_2700_arc_coeff_k4= -1.557e-12
im_2700_arc_coeff_k5= 4.556e-10
im_2700_arc_coeff_k6= -4.186e-8
im_2700_arc_coeff_k7= 8.346e-7
im_2700_arc_coeff_k8= 5.482e-5
im_2700_arc_coeff_k9= -0.003191
im_2700_arc_coeff_k10= 0.9729


exp_2700 =  im_2700_arc_coeff_k1 
print(f"k1                         = {exp_2700:.6f}")

exp_2700 += im_2700_arc_coeff_k2 * math.log10(Ibf)  
print(f"+ k2*lg(Ibf)                 = {exp_2700:.6f}")

exp_2700 += im_2700_arc_coeff_k3 * math.log10(G)
print(f"+ k3*lg(Ibf)          = {exp_2700:.6f}")

print(f"10^exponent = {10**exp_2700:.6f}")



# THE POLYNOMIAL - Current (buggy?) implementation
print("\n--- POLYNOMIAL (Hand implementation) ---")
f_ibf_2700 = (
    im_2700_arc_coeff_k4 * Ibf**6 +
    im_2700_arc_coeff_k5 * Ibf**5 +
    im_2700_arc_coeff_k6 * Ibf**4 +
    im_2700_arc_coeff_k7 * Ibf**3 +
    im_2700_arc_coeff_k8 * Ibf**2 +
    im_2700_arc_coeff_k9 * Ibf +
    im_2700_arc_coeff_k10 
)
print(f"k4*Ibf^6 = {im_2700_arc_coeff_k4 * Ibf**6:.6f}")
print(f"k5*Ibf^5 = {im_2700_arc_coeff_k5 * Ibf**5:.6f}")
print(f"k6*Ibf^4 = {im_2700_arc_coeff_k6 * Ibf**4:.6f}")
print(f"k7*Ibf^3 = {im_2700_arc_coeff_k7 * Ibf**3:.6f}")
print(f"k8*Ibf^2 = {im_2700_arc_coeff_k8 * Ibf**2:.6f}")
print(f"k9*Ibf = {im_2700_arc_coeff_k9 * Ibf:.6f}")
print(f"k10  = {im_2700_arc_coeff_k10}")
print(f"\nTotal polynomial f(Ibf) = {f_ibf_2700:.6f}")



print("\n" + "=" * 70)
print(f"result = (10^exp)*polynomial =,{(10**exp_2700)*f_ibf_2700}")
print("\n" + "=" * 70)
im_2700_result = (10**(exp_2700)) * f_ibf_2700
print(f"Intermeddiate Arcing Current at 600V = {im_2700_result:2.3f} kA")



# print(f"Intermeddiate Arcing Current at 600V = {im_result:2.3f} kA")
print(f"Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {im_2700_result:2.3f} kA (expected: {expected['Iarc_2700']} kA)")
lib_calc_im_2700_result = equation_1_intermediate_arcing_current(Ibf,G,ElectrodeConfig.VCB,2700)
print(f"Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: {lib_calc_im_2700_result:2.3f} kA (expected: {expected['Iarc_2700']} kA)")

IEEE IMPLEMENTATION FOR ARCING CURRENT AT 2700V - Exponent Breakdown
k1                         = 0.006500
+ k2*lg(Ibf)                 = 1.183767
+ k3*lg(Ibf)          = 1.135359
10^exponent = 13.657102

--- POLYNOMIAL (Hand implementation) ---
k4*Ibf^6 = -0.000018
k5*Ibf^5 = 0.000346
k6*Ibf^4 = -0.002119
k7*Ibf^3 = 0.002817
k8*Ibf^2 = 0.012334
k9*Ibf = -0.047865
k10  = 0.9729

Total polynomial f(Ibf) = 0.938395

result = (10^exp)*polynomial =,12.815760907648627

Intermeddiate Arcing Current at 600V = 12.816 kA
Hand Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 12.816 kA (expected: 12.816 kA)
Library Calculation for Intermediate Arcing Current @ 600V, Iarc_600: 12.816 kA (expected: 12.816 kA)


The library implementations are all correct, this means that the we were using 15000 as Ibf previously, which is the wrong format. Equation 1 in section 4.4 requires the bolted fault current to be specified in kA

# 4. Caclulating Final Arcing Currents using method outlined in Section 4.9

The final arcing current is calculated by interpolating between the calculated intermeddiate arcing currents form the previous sections using the formulae below.

$$
I_{\mathrm{arc\_1}} =
\frac{I_{\mathrm{arc\_2700}} - I_{\mathrm{arc\_600}}}{2.1}
\left( V_{\mathrm{oc}} - 2.7 \right)
+ I_{\mathrm{arc\_2700}}
$$

$$
I_{\mathrm{arc\_2}} =
\frac{I_{\mathrm{arc\_14300}} - I_{\mathrm{arc\_2700}}}{11.6}
\left( V_{\mathrm{oc}} - 14.3 \right)
+ I_{\mathrm{arc\_14300}}
$$

$$
I_{\mathrm{arc\_3}} =
\frac{I_{\mathrm{arc\_1}} (2.7 - V_{\mathrm{oc}})}{2.1}
+
\frac{I_{\mathrm{arc\_2}} (V_{\mathrm{oc}} - 0.6)}{2.1}
$$

where:

- $I_{\mathrm{arc\_1}}$ is the first $I_{\mathrm{arc}}$ interpolation term between 600 V and 2700 V (kA)
- $I_{\mathrm{arc\_2}}$ is the second $I_{\mathrm{arc}}$ interpolation term used when $V_{\mathrm{oc}} > 2700$ V (kA)
- $I_{\mathrm{arc\_3}}$ is the third $I_{\mathrm{arc}}$ interpolation term used when $V_{\mathrm{oc}} < 2700$ V (kA)
- $V_{\mathrm{oc}}$ is the open-circuit (system) voltage (kV)

When $0.600 < V_{\mathrm{oc}} \le 2.7$, the final value of arcing current is:

$$
I_{\mathrm{arc}} = I_{\mathrm{arc\_3}}
$$

When $V_{\mathrm{oc}} > 2.7$, the final value of arcing current is:

$$
I_{\mathrm{arc}} = I_{\mathrm{arc\_2}}
$$


In [ ]:
#by hand
I_arc_1 = ((im_2700_result - im_600_result)/ 2.1) * (Voc - 2.7) + im_2700_result
I_arc_2 = ((im_14300_result - im_2700_result)/ 11.6) * (Voc - 14.3) + im_14300_result
I_arc_3 = ((I_arc_1 * (2.7-Voc))/2.1) + ((I_arc_2 * (Voc - 0.6))/2.1)

hc_Iarc = I_arc_2
lib_Iarc = equations_16_17_18_mv_interpolation(lib_calc_im_600_result,lib_calc_im_2700_result,lib_calc_im_14300_result,Voc)

#TODO Add expected values and the other values that are calculated for this section of Annex D1 (although they are not used) in the pytest file later on.
print(f"Hand Calculation: The final arcing currents is {hc_Iarc:2.3f} kA (expected: 12.979 kA)")
print(f"Library Calculation: The final arcing currents is {lib_Iarc:2.3f} kA (expected: 12.979 kA)")

#REF STD TO SEE THAT T=197ms on a typical MV TCC

Hand Calculation: The final arcing currents is 12.979 kA (expected: 12.979 kA)
Library Calculation: The final arcing currents is 12.979 kA (expected: 12.979 kA)


# 5. Calculating Enclosure Size Correction Factor

Finding the enclosure size correction factor using the equations and instructions from Section 4.8

In [ ]:
adj_width = (660.4 + (enc_width - 660.4) * ((Voc + 4)/20))*25.4**-1
print(f"Adjusted width = {adj_width:2.3f} (expected 27.632)")

Adjusted width = 27.632 (expected 27.632)


In [ ]:
adj_height = 0.03937*enc_height
print(f"Adjusted width = {adj_height:2.3f} (expected 45)")

Adjusted width = 45.000 (expected 45)


In [ ]:
#calculating the equivalent box size
ees = (adj_height + adj_width)/2
print(f"Equivalent Enclosure Size = {ees:2.3f} (expected 36.316)")

Equivalent Enclosure Size = 36.316 (expected 36.316)


In [ ]:
#calculating correction factor
cf = -0.000302*ees**2 + 0.03441*ees + 0.4325

lib_cf = equation_13_equivalent_enclosure_size(enc_height,enc_width,Voc,config,is_shallow=False) #The example uses the formula for Typical Enclosure, that is, is_shallow=False. Also the width >508mm

print(f"Hand Calculation: Correction Factor = {cf:2.3f} (expected 1.284)")
print(f"Library Calculation: Correction Factor = {lib_cf:2.3f} (expected 1.284)")


Hand Calculation: Correction Factor = 1.284 (expected 1.284)
Library Calculation: Correction Factor = 37.431 (expected 1.284)


# 6. Calculating the intermediate values of Incident Energy

Using coeeficients from Table 3,4 and 5 with Equation (3), (4) and (5), we find the intermediate incident energies.

In [ ]:
#TODO IMPLEMENT HAND CALCULATION AND COMPARE AGAINST LIB FOR PG 60 OF STD.

## x. Once Fixed: Full Validation

After updating `incident_energy.py`, run this to validate:

In [ ]:
# #After fixing the source code, test the full calculation
# result = calculate_arc_flash(
#     ibf_ka=Ibf,
#     voc_kv=Voc,
#     gap_mm=G,
#     working_distance_mm=D,
#     arc_duration_ms=T,
#     height_mm=508.0,
#     width_mm=508.0,
#     depth_mm=508.0,
#     config=config
# )

# print(f"Iarc: {result.iarc_ka:.2f} kA (expected: {expected['Iarc_final']})")
# print(f"E: {result.incident_energy_j_cm2:.2f} J/cm² (expected: {expected['E_final']})")

Iarc: 12.98 kA (expected: 12.979)
E: 2064005459514905.50 J/cm² (expected: 12.152)
